<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20SCHEMA%20NORMALIZATION/Step6_Normalize_CrossVQA_Parquet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 6: End-to-End Dataset Normalization & Parquet Conversion (CrossVQA)

This notebook normalizes **TextVQA** (`gamusa/textvqa-v0.5.1`) and **DocVQA** (`gamusa/docvqa-task1-spdocvqa`) into the **Canonical CrossVQA Schema**:
1. **Harmonized Splits:** Standardizes to `train`, `validation`, and `test`.
2. **Spatial Normalization:** Scales all OCR bounding boxes (Rosetta & Microsoft OCR) to `[ymin, xmin, ymax, xmax] ∈ [0, 1000]`.
3. **Visual Quality:** Auto-corrects EXIF rotation for TextVQA.
4. **Parquet Serialization:** Serializes into columnar Parquet files with embedded images for instant Hugging Face Dataset Viewer browsing.
5. **Hub Deployment:** Uploads canonical datasets to Hugging Face.

In [ ]:
# 1. Environment Setup & HF Authentication
!pip install -q --upgrade huggingface_hub datasets pyarrow pandas tqdm Pillow

import os
import io
import json
import zipfile
import tarfile
from collections import Counter
from pathlib import Path
from typing import List, Dict, Any, Optional
from PIL import Image, ImageOps
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login, hf_hub_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("Please configure HF_TOKEN in Colab Secrets or environment variables!")

login(token=HF_TOKEN, write_permission=True)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
print(f"✓ Authenticated as: {username}")

# Workspace paths
WORK_DIR = Path("./crossvqa_workspace")
RAW_DIR = WORK_DIR / "raw"
NORM_DIR = WORK_DIR / "normalized"
RAW_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Canonical Helper Functions
def extract_canonical_answer(answers: Optional[List[str]]) -> Optional[str]:
    if not answers:
        return None
    cleaned = [ans.strip() for ans in answers if ans and ans.strip()]
    if not cleaned:
        return None
    return Counter(cleaned).most_common(1)[0][0]

def normalize_rosetta_box(box_dict: Dict[str, float]) -> List[int]:
    tl_x = box_dict.get("top_left_x", 0.0)
    tl_y = box_dict.get("top_left_y", 0.0)
    w = box_dict.get("width", 0.0)
    h = box_dict.get("height", 0.0)
    xmin = int(round(max(0.0, min(1.0, tl_x)) * 1000))
    ymin = int(round(max(0.0, min(1.0, tl_y)) * 1000))
    xmax = int(round(max(0.0, min(1.0, tl_x + w)) * 1000))
    ymax = int(round(max(0.0, min(1.0, tl_y + h)) * 1000))
    return [ymin, xmin, ymax, xmax]

def normalize_ms_ocr_polygon(polygon: List[float], img_w: int, img_h: int) -> List[int]:
    if not polygon or len(polygon) < 8 or img_w <= 0 or img_h <= 0:
        return [0, 0, 0, 0]
    xs = polygon[0::2]
    ys = polygon[1::2]
    xmin = int(round(max(0.0, min(1.0, min(xs) / img_w)) * 1000))
    xmax = int(round(max(0.0, min(1.0, max(xs) / img_w)) * 1000))
    ymin = int(round(max(0.0, min(1.0, min(ys) / img_h)) * 1000))
    ymax = int(round(max(0.0, min(1.0, max(ys) / img_h)) * 1000))
    return [ymin, xmin, ymax, xmax]

In [ ]:
# 3. Download TextVQA Assets from Hub
TEXTVQA_REPO = f"{username}/textvqa-v0.5.1"
print(f"Downloading TextVQA assets from {TEXTVQA_REPO}...")

textvqa_files = [
    "annotations/TextVQA_0.5.1_train.json",
    "annotations/TextVQA_0.5.1_val.json",
    "annotations/TextVQA_0.5.1_test.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_train.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_val.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_test.json",
    "images/train_val_images.zip",
    "images/test_images.zip",
]

textvqa_paths = {}
for rfile in textvqa_files:
    textvqa_paths[rfile] = hf_hub_download(
        repo_id=TEXTVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "textvqa")
    )
print("✓ TextVQA assets downloaded successfully.")

In [ ]:
# 4. Normalize and Export TextVQA to Parquet
def process_textvqa(split_name, ann_key, ocr_key, zip_key):
    ann_path = textvqa_paths[ann_key]
    ocr_path = textvqa_paths[ocr_key]
    zip_path = textvqa_paths[zip_key]
    out_path = NORM_DIR / f"textvqa_{split_name}.parquet"
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']
    with open(ocr_path, 'r', encoding='utf-8') as f:
        ocr_raw = json.load(f)['data']
        
    ocr_lookup = {}
    for o in ocr_raw:
        tokens = o.get('ocr_tokens', [])
        boxes = [normalize_rosetta_box(b.get('bounding_box', {})) for b in o.get('ocr_info', [])]
        ocr_lookup[o['image_id']] = {'tokens': tokens, 'boxes': boxes}
        
    zip_ref = zipfile.ZipFile(zip_path, 'r')
    prefix = "train_images/" if split_name != "test" else "test_images/"
    
    rows = []
    for item in tqdm(items, desc=f"TextVQA [{split_name}]"):
        q_id = str(item['question_id'])
        img_id = item['image_id']
        img_name = f"{prefix}{img_id}.jpg"
        
        raw_bytes = zip_ref.read(img_name)
        img = ImageOps.exif_transpose(Image.open(io.BytesIO(raw_bytes)))
        orig_w, orig_h = img.size
        
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95)
        
        ocr_entry = ocr_lookup.get(img_id, {'tokens': [], 'boxes': []})
        answers = item.get('answers', None)
        
        rows.append({
            "sample_id": f"textvqa_{q_id}",
            "domain_type": "scene_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item.get('question_tokens', []),
            "image_id": img_id,
            "image": {'bytes': buf.getvalue(), 'path': f"{img_id}.jpg"},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_entry['tokens'],
            "ocr_boxes": ocr_entry['boxes'],
            "metadata": json.dumps({
                "image_classes": item.get('image_classes', []),
                "flickr_original_url": item.get('flickr_original_url', '')
            }, ensure_ascii=False)
        })
    zip_ref.close()
    
    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False, engine='pyarrow', compression='zstd')
    print(f"✓ Exported {out_path} ({len(df)} rows, {out_path.stat().st_size / (1024**2):.1f} MB)")

process_textvqa("train", "annotations/TextVQA_0.5.1_train.json", "ocr/TextVQA_Rosetta_OCR_v0.2_train.json", "images/train_val_images.zip")
process_textvqa("validation", "annotations/TextVQA_0.5.1_val.json", "ocr/TextVQA_Rosetta_OCR_v0.2_val.json", "images/train_val_images.zip")
process_textvqa("test", "annotations/TextVQA_0.5.1_test.json", "ocr/TextVQA_Rosetta_OCR_v0.2_test.json", "images/test_images.zip")

In [ ]:
# 5. Download DocVQA Assets from Hub
DOCVQA_REPO = f"{username}/docvqa-task1-spdocvqa"
print(f"Downloading DocVQA assets from {DOCVQA_REPO}...")

docvqa_files = [
    "annotations/train_v1.0_withQT.json",
    "annotations/val_v1.0_withQT.json",
    "annotations/test_v1.0.json",
    "images/spdocvqa_images.tar.gz",
    "ocr/spdocvqa_ocr.tar.gz"
]

docvqa_paths = {}
for rfile in docvqa_files:
    docvqa_paths[rfile] = hf_hub_download(
        repo_id=DOCVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "docvqa")
    )
print("✓ DocVQA assets downloaded successfully.")

In [ ]:
# 6. Normalize and Export DocVQA to Parquet
def process_docvqa(split_name, ann_key):
    ann_path = docvqa_paths[ann_key]
    img_tar_path = docvqa_paths["images/spdocvqa_images.tar.gz"]
    ocr_tar_path = docvqa_paths["ocr/spdocvqa_ocr.tar.gz"]
    out_path = NORM_DIR / f"docvqa_{split_name}.parquet"
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']
        
    img_tar = tarfile.open(img_tar_path, 'r:*')
    ocr_tar = tarfile.open(ocr_tar_path, 'r:*')
    
    rows = []
    for item in tqdm(items, desc=f"DocVQA [{split_name}]"):
        q_id = str(item['questionId'])
        img_rel = item['image'].lstrip("./")
        
        # Extract image
        try:
            f_img = img_tar.extractfile(img_rel)
            raw_bytes = f_img.read()
        except KeyError:
            f_img = img_tar.extractfile(os.path.basename(img_rel))
            raw_bytes = f_img.read()
            
        img = Image.open(io.BytesIO(raw_bytes))
        orig_w, orig_h = img.size
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        
        # Extract OCR
        base_id = os.path.splitext(os.path.basename(img_rel))[0]
        ocr_tokens, ocr_boxes = [], []
        for c_path in [f"ocr_results/{base_id}.json", f"{base_id}.json"]:
            try:
                f_ocr = ocr_tar.extractfile(c_path)
                ocr_json = json.loads(f_ocr.read().decode('utf-8'))
                for page in ocr_json.get('recognitionResults', []):
                    for line in page.get('lines', []):
                        for w in line.get('words', []):
                            t = w.get('text', '').strip()
                            poly = w.get('boundingBox', [])
                            if t and poly:
                                ocr_tokens.append(t)
                                ocr_boxes.append(normalize_ms_ocr_polygon(poly, orig_w, orig_h))
                break
            except KeyError:
                continue
                
        answers = item.get('answers', None)
        
        rows.append({
            "sample_id": f"docvqa_{q_id}",
            "domain_type": "document_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item['question'].lower().split(),
            "image_id": os.path.basename(img_rel),
            "image": {'bytes': buf.getvalue(), 'path': os.path.basename(img_rel)},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_tokens,
            "ocr_boxes": ocr_boxes,
            "metadata": json.dumps({
                "question_types": item.get('question_types', []),
                "docId": item.get('docId', None),
                "ucsf_document_id": item.get('ucsf_document_id', ''),
                "ucsf_document_page_no": item.get('ucsf_document_page_no', '')
            }, ensure_ascii=False)
        })
        
    img_tar.close()
    ocr_tar.close()
    
    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False, engine='pyarrow', compression='zstd')
    print(f"✓ Exported {out_path} ({len(df)} rows, {out_path.stat().st_size / (1024**2):.1f} MB)")

process_docvqa("train", "annotations/train_v1.0_withQT.json")
process_docvqa("validation", "annotations/val_v1.0_withQT.json")
process_docvqa("test", "annotations/test_v1.0.json")

In [ ]:
# 7. Verification of Normalized Splits
from datasets import load_dataset

print("=== VERIFYING TEXTVQA NORMALIZED ===")
ds_text = load_dataset('parquet', data_files={
    'train': str(NORM_DIR / 'textvqa_train.parquet'),
    'validation': str(NORM_DIR / 'textvqa_validation.parquet'),
    'test': str(NORM_DIR / 'textvqa_test.parquet')
})
print(ds_text)
assert len(ds_text['train']) == 34602, f"TextVQA train count mismatch: {len(ds_text['train'])}"
assert len(ds_text['validation']) == 5000, f"TextVQA val count mismatch: {len(ds_text['validation'])}"
assert len(ds_text['test']) == 5734, f"TextVQA test count mismatch: {len(ds_text['test'])}"
print("✓ TextVQA split counts verified: 100% Match!")

print("\n=== VERIFYING DOCVQA NORMALIZED ===")
ds_doc = load_dataset('parquet', data_files={
    'train': str(NORM_DIR / 'docvqa_train.parquet'),
    'validation': str(NORM_DIR / 'docvqa_validation.parquet'),
    'test': str(NORM_DIR / 'docvqa_test.parquet')
})
print(ds_doc)
assert len(ds_doc['train']) == 39463, f"DocVQA train count mismatch: {len(ds_doc['train'])}"
assert len(ds_doc['validation']) == 5349, f"DocVQA val count mismatch: {len(ds_doc['validation'])}"
assert len(ds_doc['test']) == 5188, f"DocVQA test count mismatch: {len(ds_doc['test'])}"
print("✓ DocVQA split counts verified: 100% Match!")

In [ ]:
# 8. Upload Canonical Repositories to Hugging Face
def upload_canonical_repo(repo_name, prefix, task_category, tags):
    full_repo = f"{username}/{repo_name}"
    api.create_repo(repo_id=full_repo, repo_type="dataset", exist_ok=True)
    
    # Write README.md with Dataset Info
    readme_content = f"""---
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*\n  - split: validation
    path: data/validation-*\n  - split: test
    path: data/test-*\ntask_categories:
- {task_category}
tags:
{tags}
---
# {repo_name}
Normalized Canonical Dataset for CrossVQA Benchmark.\n"""
    readme_path = WORK_DIR / f"README_{repo_name}.md"
    readme_path.write_text(readme_content, encoding='utf-8')
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo="README.md",
        repo_id=full_repo,
        repo_type="dataset"
    )
    
    # Upload Parquet files
    for split in ["train", "validation", "test"]:
        p_file = NORM_DIR / f"{prefix}_{split}.parquet"
        api.upload_file(
            path_or_fileobj=str(p_file),
            path_in_repo=f"data/{split}-00000.parquet",
            repo_id=full_repo,
            repo_type="dataset"
        )
    print(f"✓ Successfully uploaded: https://huggingface.co/datasets/{full_repo}")

# Upload TextVQA Canonical
upload_canonical_repo(
    repo_name="textvqa-canonical",
    prefix="textvqa",
    task_category="visual-question-answering",
    tags="- textvqa\n- scene-text\n- canonical\n- crossvqa"
)

# Upload DocVQA Canonical
upload_canonical_repo(
    repo_name="docvqa-canonical",
    prefix="docvqa",
    task_category="visual-question-answering",
    tags="- docvqa\n- document-ai\n- canonical\n- crossvqa"
)